In [6]:
import pandas as pd
import csv
import numpy as np
from pathlib import Path

In [5]:

verbatim = pd.read_csv('/home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/verbatim.txt',
                 sep='\t', 
                 engine='python',  # More flexible parser
                 on_bad_lines='skip', nrows=10,
                 quoting=csv.QUOTE_NONE)

In [9]:
!wc -l /home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/verbatim.txt

117104004 /home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/verbatim.txt


In [ ]:
list(verbatim.columns)

In [16]:
row = verbatim.iloc[-1].to_dict()

In [18]:
for k, v in row.items():
    if isinstance(v, np.ndarray):
        print(k)
        

In [ ]:
multimedia = pd.read_csv('/home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/multimedia.txt',
                 sep='\t', 
                 engine='python',  # More flexible parser
                 on_bad_lines='skip', usecols=["type","identifier"],
                 quoting=csv.QUOTE_NONE)

In [ ]:
multimedia.columns

In [ ]:
multimedia.type.unique()

# Pre-existing

In [7]:
inat_audios_in_bucket = []
with open("inaturalist_audio.txt", "r") as f:
    for line in f.readlines():
        inat_audios_in_bucket.append(Path(line.strip()).stem)

In [8]:
len(inat_audios_in_bucket)

256714

In [18]:
inat_audios_in_bucket[:5]

['10', '10000', '100000', '100002', '100003']

In [19]:
from pathlib import Path
from urllib.parse import urlparse
import pdb

In [20]:
def _get_fname_from_url(url: str):
    parsed = urlparse(url)
    return Path(parsed.path).stem

In [23]:
to_download = []

In [ ]:
tot_num = 0
for i, chunk in enumerate(pd.read_csv('/home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/multimedia.txt',
                 sep='\t', 
                 engine='python',  # More flexible parser
                 on_bad_lines='skip', usecols=["type","identifier"],
                 quoting=csv.QUOTE_NONE, chunksize=1_000_000)):
    
    subchunk = chunk[chunk["type"] == "Sound"]
    if not subchunk.empty:
        fnames = subchunk["identifier"].apply(_get_fname_from_url)
        fnames_bool = fnames.isin(inat_audios_in_bucket).to_numpy()

        if fnames_bool.sum() > 0:
            print(f"Found {fnames_bool.sum()} overlapping files")
            
        ids = np.where(fnames_bool == False)[0]
        fnames = subchunk["identifier"].iloc[ids].tolist()
        to_download.extend(fnames)

    if len(to_download) >= 100_000:
        tot_num += len(to_download)
        print(f"Current num of files to download = {tot_num}")
        inat_to_download = pd.DataFrame({"identifier": to_download}).to_json("inat_to_download.jsonl", orient="records",lines=True, mode="a")
        to_download = []

In [9]:
chunk.head(5)

,type,identifier
0,StillImage,https://inaturalist-open-data.s3.amazonaws.com...
1,StillImage,https://inaturalist-open-data.s3.amazonaws.com...
2,StillImage,https://inaturalist-open-data.s3.amazonaws.com...
3,StillImage,https://inaturalist-open-data.s3.amazonaws.com...
4,StillImage,https://inaturalist-open-data.s3.amazonaws.com...


In [25]:
inat_to_download = pd.read_json("inat_to_download.jsonl", orient="records",lines=True,)
inat_to_download.shape

(406510, 1)

In [13]:
inat_to_download.head(5)

,identifier
0,https://static.inaturalist.org/sounds/17376.m4...
1,https://static.inaturalist.org/sounds/19469.mp...
2,https://static.inaturalist.org/sounds/23246.mp...
3,https://static.inaturalist.org/sounds/23384.mp...
4,https://static.inaturalist.org/sounds/29118.wa...


In [19]:
inat_to_download.head(5).to_dict()

{'identifier': {0: 'https://static.inaturalist.org/sounds/17376.m4a?1516933705',
  1: 'https://static.inaturalist.org/sounds/19469.mp3?1523741395',
  2: 'https://static.inaturalist.org/sounds/23246.mp3?1529475596',
  3: 'https://static.inaturalist.org/sounds/23384.mp3?1529678795',
  4: 'https://static.inaturalist.org/sounds/29118.wav?1547207371'}}

In [18]:
inat_to_download.identifier.apply(lambda x: Path(x).suffix).unique()

array(['.m4a?1516933705', '.mp3?1523741395', '.mp3?1529475596', ...,
       '.wav?1684085788', '.wav?1684111954', '.m4a?1660127046'],
      shape=(595470,), dtype=object)

# Double check that files downloaded are not already on bucket

In [10]:
import os
inat_downloaded = os.listdir("../inat_audio_downloads/")
len(inat_downloaded)

362500

In [11]:
extensions = set([Path(x).suffix for x in inat_downloaded])
extensions

{'.m4a', '.mp3', '.mpga', '.wav'}

In [12]:
stems_downloaded = [Path(x).stem for x in inat_downloaded]

In [13]:
len(set(stems_downloaded).difference(set(inat_downloaded)))

362500

In [14]:
import librosa

In [18]:
to_resample = pd.DataFrame(inat_downloaded, columns=["files"])
to_resample.head(3)

,files
0,1047641.m4a
1,502885.m4a
2,1021201.wav


In [19]:
to_resample.to_csv("../inat_to_resample_and_copy.csv", index=False)